# 03 · Text normalization of device, browser and email strings

The identity table stores user-agent-like strings. The same phone appears as `SM-G930V Build/NRD90M`,
`SAMSUNG SM-G930V` and `SM-G930V`; the same webmail provider as `hotmail.com`, `hotmail.es`, `outlook.com`
and `live.com.mx`. As raw categories each spelling is its own rare level, which splits the signal.

`fraud.text_norm` treats this as a normalization problem: casefold, strip noise tokens (Android build IDs),
map surface variants to a canonical form (brand / family / provider), and keep the informative remainder
(major version, country TLD) as separate features.

In [ ]:
import os
import altair as alt
import numpy as np
import pandas as pd

from fraud import charts, db  # charts registers the project's Altair theme
from fraud.config import get_paths
from fraud.io import load_json

PROFILE = os.environ.get("FRAUD_PROFILE", "full")  # "sample" = synthetic data
paths = get_paths(PROFILE)
pd.set_option("display.max_columns", 50, "display.width", 160)
print("profile:", PROFILE)

In [ ]:
from fraud import text_norm
cov = load_json(paths.reports / "text_norm_coverage.json")
pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "top_unmapped"} for k, v in cov.items()}).T

## Examples: many surface forms, one canonical form

In [ ]:
conn = db.connect(paths.db, read_only=True)
dev = conn.execute("SELECT DeviceInfo AS raw, count(*) AS n, avg(isFraud) AS fraud_rate FROM base "
                   "WHERE DeviceInfo IS NOT NULL GROUP BY 1 ORDER BY n DESC LIMIT 400").df()
dev = dev.join(pd.DataFrame([vars(text_norm.normalize_device(r)) for r in dev.raw]))
dev[dev.device_brand == "samsung"].head(15)

In [ ]:
email = conn.execute("SELECT P_emaildomain AS raw, count(*) AS n, avg(isFraud) AS fraud_rate FROM base "
                     "WHERE P_emaildomain IS NOT NULL GROUP BY 1 ORDER BY n DESC").df()
email = email.join(pd.DataFrame([vars(text_norm.normalize_email(r)) for r in email.raw]))
email.groupby("email_provider").agg(raw_spellings=("raw", "count"), n=("n", "sum")).sort_values("n", ascending=False).head(12)

## What is still unmapped?

In [ ]:
{k: v["top_unmapped"] for k, v in cov.items()}

## Fraud rate by normalized brand / provider

In [ ]:
seg = conn.execute("SELECT device_brand, count(*) AS n, avg(isFraud) AS fraud_rate FROM features "
                   "WHERE device_brand IS NOT NULL GROUP BY 1 HAVING count(*) >= 100 ORDER BY fraud_rate DESC").df()
seg

## Does it help the model? (ablation)

In [ ]:
res = load_json(paths.reports / "train_results.json")
pol = load_json(paths.reports / "policy_results.json")
cost = {r["model"]: r["total_cost"] for r in pol["by_model"]}
from fraud.report import MODEL_LABELS
rows = []
for k in ("lgbm", "abl_raw_text"):
    v = res["experiments"][k]
    rows.append({"variant": MODEL_LABELS[k], "valid PR-AUC": v["valid"]["pr_auc"], "test PR-AUC": v["test"]["pr_auc"],
                 "n features": v.get("n_features"), "three-tier policy cost": cost.get(MODEL_LABELS[k])})
pd.DataFrame(rows)

**Result (DRAFT, rewrite after running):** _normalization collapses N raw device strings into M families with
X% row coverage. On the model it moves PR-AUC by ... and policy cost by .... Report this honestly even if the
gain is small: gradient boosting can partly learn raw categories on its own, so the bigger wins are fewer
levels (more robust to unseen spellings in production) and features that are explainable in reason codes._